# Smart Waste Classifier: Industrial-Grade Research and Scalable Development

**Project Identifier:** Introduction to Machine Learning Module Summative  
**Objective:** Implementation of a robust, interpretable, and production-ready Computer Vision pipeline for urban waste classification using State-of-the-Art (SOTA) Transfer Learning architectures and comprehensive statistical visualization.

---

## 1. Environment Initialization and Hardware Configuration
Initialization of required libraries and hardware acceleration detection (GPU/TPU) to ensure scalable training performance.

In [ ]:
import os
import sys
import time
import logging
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve

# Global Configuration
sns.set_theme(style="whitegrid")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"Physical GPUs detected: {len(gpus)}")
else:
    print("No GPU detected. Training will proceed on CPU.")

## 2. Model Profiling and Efficiency Metrics
Analyzing the model's computational footprint, including parameter count and inference latency on the target hardware.

In [ ]:
def profile_model(model):
    total_params = model.count_params()
    trainable_params = np.sum([np.prod(v.get_shape().as_list()) for v in model.trainable_variables])
    non_trainable_params = total_params - trainable_params
    
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")
    print(f"Non-trainable Parameters: {non_trainable_params:,}")
    
    # Latency Benchmark
    dummy_input = tf.random.uniform((1, 224, 224, 3))
    start_time = time.time()
    for _ in range(100):
        _ = model(dummy_input, training=False)
    latency = (time.time() - start_time) / 100
    print(f"Average Inference Latency: {latency*1000:.2f} ms per image")

# profile_model(model)

## 3. Automated Error Analysis: Failure Case Gallery
Systematically identifying images where the model has high confidence but incorrect predictions. This is critical for debugging model bias.

In [ ]:
def analyze_failure_cases(model, dataset, class_names, num_samples=8):
    failures = []
    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        true_idx = np.argmax(labels.numpy(), axis=1)
        pred_idx = np.argmax(preds, axis=1)
        confidences = np.max(preds, axis=1)
        
        for i in range(len(true_idx)):
            if true_idx[i] != pred_idx[i]:
                failures.append({
                    "image": images[i].numpy().astype("uint8"),
                    "true": class_names[true_idx[i]],
                    "pred": class_names[pred_idx[i]],
                    "conf": confidences[i]
                })
    
    # Sort by confidence (highest confidence failures first)
    failures = sorted(failures, key=lambda x: x['conf'], reverse=True)
    
    plt.figure(figsize=(16, 8))
    for i in range(min(num_samples, len(failures))):
        plt.subplot(2, 4, i + 1)
        plt.imshow(failures[i]['image'])
        plt.title(f"True: {failures[i]['true']}\nPred: {failures[i]['pred']} ({failures[i]['conf']:.2f})")
        plt.axis('off')
    plt.suptitle("High-Confidence Failure Analysis")
    plt.show()

## 4. Optimal Threshold Tuning (Precision-Recall Calibration)
For critical categories like 'Hazardous' waste, we may prefer higher precision over recall. This cell calculates the optimal classification thresholds.

In [ ]:
def plot_precision_recall_vs_threshold(y_true, y_probs, class_idx, class_name):
    precisions, recalls, thresholds = precision_recall_curve(y_true[:, class_idx], y_probs[:, class_idx])
    
    plt.figure(figsize=(10, 6))
    plt.plot(thresholds, precisions[:-1], "b--", label="Precision")
    plt.plot(thresholds, recalls[:-1], "g-", label="Recall")
    plt.title(f"Precision-Recall Threshold Tradeoff: {class_name}")
    plt.xlabel("Probability Threshold")
    plt.legend()
    plt.grid(True)
    plt.show()

# plot_precision_recall_vs_threshold(y_true, y_probs, 0, "Hazardous")

## 5. Deployment Serialization and Signature Mapping
Defining the concrete input/output signatures for TensorFlow Serving to ensure the model integrates seamlessly with the FastAPI production environment.

In [ ]:
@tf.function(input_signature=[tf.TensorSpec([None, 224, 224, 3], tf.float32)])
def serving_fn(images):
    # Standardized preprocessing if required
    return {"predictions": model(images, training=False)}

def save_for_serving(model, export_path):
    tf.saved_model.save(
        model, 
        export_path, 
        signatures={'serving_default': serving_fn}
    )
    print(f"Model exported with standard serving signatures to {export_path}")

# save_for_serving(model, "../models/waste_classifier_serving/1")

## 6. Post-Training Quantization (Edge Optimization)
Converting the model to a float16 or integer-quantized format to reduce binary size and increase speed on mobile/edge devices.

In [ ]:
def convert_to_tflite(model, output_path):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    # Optional: converter.target_spec.supported_types = [tf.float16]
    tflite_model = converter.convert()
    
    with open(output_path, "wb") as f:
        f.write(tflite_model)
    print(f"Optimized TFLite model generated at {output_path} (Size: {len(tflite_model)/1024/1024:.2f} MB)")

# convert_to_tflite(model, "../models/waste_classifier_edge.tflite")

---